# Usar el paquete `sentiment_bias` directamente (sin FastAPI)

Este notebook usa el paquete instalado, sin pasar por el servidor HTTP y sin
hacks de `sys.path`. Requiere haberlo instalado antes:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -e .
```

**Funciones principales:**
- `sb.fit(data, n_dimensions=...)` — estima los parámetros latentes a partir de menciones.
- `sb.simulate(z, a, b, n_mentions=...)` — genera menciones sintéticas a partir de parámetros conocidos.
- `sb.thin(tensor, fraction)` — ralea un tensor a una fracción de sus menciones.

In [1]:
import numpy as np
import pandas as pd
import sentiment_bias as sb

sb.__version__

'0.1.0'

---
## 1. Analizar datos (`sb.fit`)

`fit` acepta lo que le pases: un `MentionTensor`, un DataFrame ancho o largo,
un array `(m, k, 3)`, o una lista de registros como la de abajo.

In [2]:
# --- Editá este bloque con tus datos ---
mentions = [
    {"outlet": "A", "subject": "X", "mention_type": "positive", "amount_of_mentions": 12},
    {"outlet": "A", "subject": "X", "mention_type": "negative", "amount_of_mentions": 3},
    {"outlet": "A", "subject": "Y", "mention_type": "negative", "amount_of_mentions": 5},
    {"outlet": "A", "subject": "Y", "mention_type": "neutral",  "amount_of_mentions": 2},
    {"outlet": "B", "subject": "X", "mention_type": "neutral",  "amount_of_mentions": 8},
    {"outlet": "B", "subject": "X", "mention_type": "positive", "amount_of_mentions": 2},
    {"outlet": "B", "subject": "Y", "mention_type": "positive", "amount_of_mentions": 7},
    {"outlet": "C", "subject": "X", "mention_type": "negative", "amount_of_mentions": 4},
    {"outlet": "C", "subject": "X", "mention_type": "positive", "amount_of_mentions": 6},
]
# ---------------------------------------

# También podés cargar desde un archivo:
#   tensor = sb.read_json("../data/input.json")
#   tensor = sb.read_csv("../data/input.csv")

tensor = sb.to_tensor(mentions)
print(tensor.shape, tensor.n_mentions, "menciones")
tensor.to_frame("wide")

(3, 2, 3) 49 menciones


,outlet,subject,negative,neutral,positive
0,A,X,3,0,12
1,A,Y,5,2,0
2,B,X,0,8,2
3,B,Y,0,0,7
4,C,X,4,0,6
5,C,Y,0,0,0


In [3]:
# n_restarts > 1 porque el objetivo no es convexo; seed lo hace reproducible
result = sb.fit(tensor, n_dimensions=1, n_restarts=10, seed=0)
result

<BiasFit D=1 outlets=3 subjects=2 loss=41.94 bic=111.1 restarts=10 converged>

In [4]:
print(result.summary())

BiasFit: D=1, 3 outlets, 2 subjects
  loss=41.9450  bic=111.13  restarts=10  converged
  49 mentions, 7 parameters

Outlet bias (dim_1), most negative to most positive:
outlet
B   -0.636847
C   -0.124091
A    0.760938

Subjects:
            dim_1         b
subject                    
X        0.530446  0.528237
Y       -2.367758  0.627996


In [5]:
# Sesgo por medio: DataFrame indexado por medio, una columna por dimensión
result.z

,dim_1
outlet,
A,0.760938
B,-0.636847
C,-0.124091


In [6]:
# Parámetros por sujeto: a (discriminación) y b (sentimiento de base)
result.subjects

,dim_1,b
subject,,
X,0.530446,0.528237
Y,-2.367758,0.627996


In [7]:
print(f"Loss (neg-log-likelihood): {result.loss:.4f}")
print(f"BIC:                       {result.bic:.4f}")
print(f"Convergió:                 {result.converged}")

# Cuántos mínimos distintos encontraron los restarts. Si hay más de uno,
# el objetivo es multimodal y conviene subir n_restarts.
print(f"Mínimos distintos:         {len(set(round(l, 6) for l in result.losses))} de {result.n_restarts}")

Loss (neg-log-likelihood): 41.9450
BIC:                       111.1327
Convergió:                 True
Mínimos distintos:         8 de 10


In [8]:
# Log-odds ajustados q_ij para cada par (medio, sujeto)
result.log_odds()

subject,X,Y
outlet,,
A,0.931874,-1.173722
B,0.190424,2.135896
C,0.462413,0.921814


### Convención de normalización (D=1)

Con `n_dimensions=1` el modelo solo está identificado a menos de una
reparametrización afín de `z`, así que la salida se canonicaliza a

$$\sum_i z_i = 0 \qquad \sum_i z_i^2 = 1$$

absorbiendo la escala en `a` y el corrimiento en `b`, de modo que todos los
`q_ij` quedan intactos. Es norma unitaria, **no** varianza unitaria: como
`sum(z²) = m · var(z)`, con media cero las dos son mutuamente excluyentes, y
bajo esta convención `var(z) = 1/m`.

El signo residual lo fija `sign_reference_outlet` (por defecto, el primer
medio alfabéticamente queda positivo).

Con `n_dimensions > 1` la salida **no** se canonicaliza.

In [9]:
z = result.z["dim_1"].to_numpy()
m = len(z)

print(f"m (medios)  = {m}")
print(f"sum(z)      = {z.sum():+.3e}   (debe ser 0)")
print(f"sum(z^2)    = {(z**2).sum():.10f}   (debe ser 1)")
print(f"var(z)      = {z.var():.10f}   (= 1/m = {1/m:.10f})")

# La canonicalización no cambia el ajuste: q_ij es invariante.
q = np.outer(z, result.a["dim_1"]) + result.b.to_numpy()
print(f"\nq_ij invariante: {np.allclose(q, result.log_odds().to_numpy())}")

m (medios)  = 3
sum(z)      = +5.551e-17   (debe ser 0)
sum(z^2)    = 1.0000000000   (debe ser 1)
var(z)      = 0.3333333333   (= 1/m = 0.3333333333)

q_ij invariante: True


---
## 2. Generar datos sintéticos (`sb.simulate`)

Pasale parámetros conocidos y generá menciones simuladas. Acá reusamos la
salida del ajuste anterior como parámetros de entrada.

In [10]:
generated = sb.simulate(
    result.z.to_numpy(),
    result.a.to_numpy(),
    result.b.to_numpy(),
    n_mentions=100,
    outlets=result.outlets,
    subjects=list(result.a.index),
    seed=0,
)
print(generated.shape, generated.n_mentions, "menciones")
generated.to_frame("wide")

(3, 2, 3) 600 menciones


,outlet,subject,negative,neutral,positive
0,A,X,11,22,67
1,A,Y,79,20,1
2,B,X,31,39,30
3,B,Y,2,11,87
4,C,X,26,32,42
5,C,Y,3,32,65


---
## 3. Ciclo completo: generar → estimar → verificar recuperación

Generá datos desde parámetros conocidos y verificá que la estimación los
recupera. Con `n_mentions="poisson"` el número de menciones por par se sortea
de una Poisson, que es más realista que un N fijo.

In [11]:
rng = np.random.default_rng(0)

# Parámetros verdaderos
z_true = np.linspace(-2, 2, 50)          # 50 medios
a_true = np.linspace(-1, 1, 6)           # 6 sujetos
b_true = rng.normal(0, 0.05, 6)

synthetic = sb.simulate(
    z_true, a_true, b_true,
    n_mentions="poisson", mean_mentions=100,
    seed=123457,
)
print(synthetic.shape, synthetic.n_mentions, "menciones")
print("Menciones por par: media",
      synthetic.counts.sum(axis=2).mean(), "desvío",
      round(synthetic.counts.sum(axis=2).std(), 2))

(50, 6, 3) 30145 menciones
Menciones por par: media 100.48333333333333 desvío 10.03


In [12]:
estimated = sb.fit(synthetic, n_dimensions=1, n_restarts=5, seed=1)
estimated

<BiasFit D=1 outlets=50 subjects=6 loss=28743.69 bic=58126.8 restarts=5 converged>

In [13]:
from scipy.stats import spearmanr

# El signo de z es convencional, así que comparamos en valor absoluto
for name, true, pred in [
    ("z", z_true, estimated.z["dim_1"].to_numpy()),
    ("a", a_true, estimated.a["dim_1"].to_numpy()),
    ("b", b_true, estimated.b.to_numpy()),
]:
    print(f"{name}: |Spearman| = {abs(spearmanr(true, pred).statistic):.4f}")

z: |Spearman| = 0.9943
a: |Spearman| = 1.0000
b: |Spearman| = 0.0286


---
## 4. Consistencia del modelo (`sb.thin`)

`thin` ralea el tensor a una fracción de sus menciones, muestreando sin
reemplazo. Sirve para medir cuánto se degrada el ajuste con menos datos.

In [14]:
for fraction in [1.0, 0.5, 0.2, 0.05]:
    reduced = sb.thin(synthetic, fraction, seed=1)
    fit_reduced = sb.fit(reduced, n_dimensions=1, n_restarts=3, seed=1)
    rho = abs(spearmanr(z_true, fit_reduced.z["dim_1"]).statistic)
    print(f"fracción {fraction:>5}  {reduced.n_mentions:>6} menciones  |Spearman(z)| = {rho:.4f}")

fracción   1.0   30145 menciones  |Spearman(z)| = 0.9943


fracción   0.5   15072 menciones  |Spearman(z)| = 0.9922


fracción   0.2    6029 menciones  |Spearman(z)| = 0.9901


fracción  0.05    1507 menciones  |Spearman(z)| = 0.9562
